# Финальная проверка клиентов за февраль

Проверки:
1) уникальность ИНН в Excel и витрине клиентов;
2) заполненность основных полей;
3) top-10 only_excel ИНН и поиск в исходных таблицах;
4) транзакции этих клиентов за февраль;
5) авто-классификация причин расхождений.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx"
excel_inn_col = "ИНН"
excel_name_col = "Наименование"
excel_agr_id_col = "ID договора"
excel_contract_col = "Номер договора"

report_month = "2026-02-01"
month_start = pd.to_datetime(report_month).strftime("%Y-%m-%d")
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d")
top_n_only_excel = 10

merchants_snapshot_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute(f"invalidate metadata {merchants_snapshot_table}")
print('Impala initialized, metadata invalidated')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def is_filled(v):
    if pd.isna(v):
        return False
    return str(v).strip() not in {'', 'nan', 'None', 'null'}

def to_sql_in_list(vals):
    safe = [str(v).replace("'", "''") for v in vals if pd.notna(v)]
    return ','.join([f"'{x}'" for x in safe])

## 0) Загрузка Excel и среза витрины клиентов (активные на 1-е)

In [ ]:
df_excel = pd.read_excel(excel_path)
for c in [excel_inn_col, excel_name_col, excel_agr_id_col, excel_contract_col]:
    if c not in df_excel.columns:
        raise ValueError(f"В Excel нет колонки: {c}")

excel_norm = pd.DataFrame({
    'inn': df_excel[excel_inn_col].apply(normalize_id),
    'excel_client_name': df_excel[excel_name_col].apply(normalize_str),
    'excel_agr_id': df_excel[excel_agr_id_col].apply(normalize_id),
    'excel_contract_number': df_excel[excel_contract_col].apply(normalize_str),
}).dropna(how='all')

with imp:
    dm = imp.fetch(f"""
        select
            cast(inn as string) as inn,
            cast(agr_id as string) as agr_id,
            cast(c_agr_number as string) as c_agr_number,
            cast(cmp_name as string) as cmp_name,
            cast(d_valid_from as date) as d_valid_from,
            cast(d_valid_to as date) as d_valid_to
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (d_valid_to is null or cast(d_valid_to as date) >= cast('{month_start}' as date))
    """)

dm_norm = pd.DataFrame({
    'inn': dm['inn'].apply(normalize_id),
    'agr_id': dm['agr_id'].apply(normalize_id),
    'c_agr_number': dm['c_agr_number'].apply(normalize_str),
    'cmp_name': dm['cmp_name'].apply(normalize_str),
}).dropna(how='all')

print('excel_rows =', len(excel_norm))
print('merchants_rows_active_on_1st =', len(dm_norm))

## 1) Уникальность ИНН

In [ ]:
excel_inn_dup = (
    excel_norm.dropna(subset=['inn'])
    .groupby('inn', as_index=False).size()
    .rename(columns={'size': 'cnt'})
)
excel_inn_dup = excel_inn_dup[excel_inn_dup['cnt'] > 1]

dm_inn_dup = (
    dm_norm.dropna(subset=['inn'])
    .groupby('inn', as_index=False).size()
    .rename(columns={'size': 'cnt'})
)
dm_inn_dup = dm_inn_dup[dm_inn_dup['cnt'] > 1]

uniq_summary = pd.DataFrame([
    {'source': 'excel', 'rows': len(excel_norm), 'unique_inn': excel_norm['inn'].dropna().nunique(), 'duplicated_inn_cnt': len(excel_inn_dup)},
    {'source': 'merchants_snapshot', 'rows': len(dm_norm), 'unique_inn': dm_norm['inn'].dropna().nunique(), 'duplicated_inn_cnt': len(dm_inn_dup)},
])
display(uniq_summary)
display(excel_inn_dup.head(100))
display(dm_inn_dup.head(100))

## 2) Заполненность полей

In [ ]:
excel_fields = ['inn', 'excel_client_name', 'excel_agr_id', 'excel_contract_number']
dm_fields = ['inn', 'cmp_name', 'agr_id', 'c_agr_number']

rows = []
for f in excel_fields:
    total = len(excel_norm)
    filled = int(excel_norm[f].apply(is_filled).sum())
    rows.append({'source': 'excel', 'field': f, 'rows': total, 'filled_cnt': filled, 'fill_rate_pct': round(100.0 * filled / total, 2) if total else 0.0})
for f in dm_fields:
    total = len(dm_norm)
    filled = int(dm_norm[f].apply(is_filled).sum())
    rows.append({'source': 'merchants_snapshot', 'field': f, 'rows': total, 'filled_cnt': filled, 'fill_rate_pct': round(100.0 * filled / total, 2) if total else 0.0})
display(pd.DataFrame(rows))

## 3) Top-10 only_excel INN и сравнение в источниках

In [ ]:
set_excel_inn = set(excel_norm['inn'].dropna().tolist())
set_dm_inn = set(dm_norm['inn'].dropna().tolist())
only_excel_inn = sorted(list(set_excel_inn - set_dm_inn))
top10_inn = only_excel_inn[:top_n_only_excel]

display(pd.DataFrame({'top10_only_excel_inn': top10_inn}))

if top10_inn:
    in_inn = to_sql_in_list(top10_inn)
    with imp:
        source_profile = imp.fetch(f"""
            select
                cast(c.c_inn as string) as inn,
                cast(c.c_cmp_name as string) as client_name_clients,
                cast(a.abs_agr_id as string) as abs_agr_id,
                cast(a.n_agr as string) as n_agr,
                cast(a.c_agr_number as string) as c_agr_number,
                cast(a.acq_class as string) as acq_class,
                cast(a.d_valid_from as date) as d_valid_from,
                cast(a.d_valid_to as date) as d_valid_to,
                cast(m.c_nmrc as string) as retl_id,
                cast(m.c_mrc_name as string) as retl_name,
                cast(m.n_mcc as string) as mcc,
                cast(r2.id as string) as r2_id
            from ods_alpha.scd1_companies c
            left join ods_alpha.scd1_agreements a
              on a.n_cmp_client = c.n_cmp
             and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
             and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            left join ods_alpha.scd1_merchants m on m.n_cmp = c.n_cmp
            left join ods.scd1_z_r2_ip_merchants r2 on cast(r2.id as string) = cast(a.abs_agr_id as string)
            where cast(c.c_inn as string) in ({in_inn})
            order by inn, c_agr_number
        """)
    display(source_profile.head(500))
    excel_top10 = excel_norm[excel_norm['inn'].isin(top10_inn)][['inn', 'excel_client_name', 'excel_agr_id', 'excel_contract_number']].drop_duplicates()
    display(excel_top10.head(500))

## 4) Транзакции top-10 only_excel INN за февраль

In [ ]:
if top10_inn:
    in_inn = to_sql_in_list(top10_inn)
    with imp:
        trx_check = imp.fetch(f"""
            with agr as (
                select
                    cast(c.c_inn as string) as inn,
                    cast(a.n_agr as string) as n_agr,
                    cast(a.abs_agr_id as string) as abs_agr_id,
                    cast(a.c_agr_number as string) as c_agr_number,
                    cast(a.acq_class as string) as acq_class
                from ods_alpha.scd1_companies c
                join ods_alpha.scd1_agreements a on a.n_cmp_client = c.n_cmp
                where cast(c.c_inn as string) in ({in_inn})
                  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
                  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
            ),
            ta_small as (
                select
                    cast(ta.n_agr as string) as n_agr,
                    ta.n_trx
                from ods_alpha.scd1_trx_acq ta
                join (select distinct n_agr from agr) a
                  on cast(ta.n_agr as string) = a.n_agr
            ),
            trx_small as (
                select
                    t.n_trx,
                    cast(t.n_amt_src as double) as n_amt_src
                from ods_alpha.scd1_trx t
                where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
                  and t.c_trx_class = 'SA'
                  and t.c_trx_type = 'S01'
                  and t.c_nter is not null
                  and t.ods_deleted_flg <> '1'
                  and t.cf_trx_stat <> 'R'
            )
            select
                a.inn,
                a.abs_agr_id,
                a.c_agr_number,
                a.acq_class,
                count(distinct t.n_trx) as trx_dag_cnt_feb,
                sum(t.n_amt_src) as trx_dag_sum_feb
            from agr a
            join ta_small ta on ta.n_agr = a.n_agr
            join trx_small t on t.n_trx = ta.n_trx
            group by a.inn, a.abs_agr_id, a.c_agr_number, a.acq_class
            order by a.inn, a.c_agr_number
        """)
    display(trx_check.head(500))
else:
    trx_check = pd.DataFrame()

## 5) Авто-классификация причин расхождений

In [ ]:
if top10_inn:
    in_inn = to_sql_in_list(top10_inn)
    with imp:
        reason_base = imp.fetch(f"""
            with companies as (
                select cast(c_inn as string) as inn, count(*) as companies_cnt
                from ods_alpha.scd1_companies
                where cast(c_inn as string) in ({in_inn})
                group by cast(c_inn as string)
            ),
            agreements as (
                select
                    cast(c.c_inn as string) as inn,
                    count(*) as agr_total_cnt,
                    sum(case when a.acq_class = 'SA' then 1 else 0 end) as agr_sa_cnt,
                    sum(case when a.acq_class = 'SA'
                               and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
                               and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
                             then 1 else 0 end) as agr_sa_active_on_1st_cnt
                from ods_alpha.scd1_companies c
                join ods_alpha.scd1_agreements a on a.n_cmp_client = c.n_cmp
                where cast(c.c_inn as string) in ({in_inn})
                group by cast(c.c_inn as string)
            )
            select
                x.inn,
                coalesce(c.companies_cnt, 0) as companies_cnt,
                coalesce(a.agr_total_cnt, 0) as agr_total_cnt,
                coalesce(a.agr_sa_cnt, 0) as agr_sa_cnt,
                coalesce(a.agr_sa_active_on_1st_cnt, 0) as agr_sa_active_on_1st_cnt
            from (select cast(val as string) as inn from (select explode(split('{in_inn}', ',')) as val) z) x
            left join companies c on c.inn = x.inn
            left join agreements a on a.inn = x.inn
        """)

    # Нормализация INN после SQL-конструкции explode/split
    reason_base['inn'] = reason_base['inn'].str.replace("'", "", regex=False)

    trx_agg = pd.DataFrame()
    if 'trx_check' in globals() and not trx_check.empty:
        trx_agg = (trx_check.groupby('inn', as_index=False)
                  .agg({'trx_dag_cnt_feb': 'sum'}))

    reasons = reason_base.merge(trx_agg, on='inn', how='left')
    reasons['trx_dag_cnt_feb'] = reasons['trx_dag_cnt_feb'].fillna(0)

    def classify(r):
        if r['companies_cnt'] == 0:
            return 'no_company_in_ods'
        if r['agr_total_cnt'] > 0 and r['agr_sa_cnt'] == 0:
            return 'only_qp_or_non_sa'
        if r['agr_sa_cnt'] > 0 and r['agr_sa_active_on_1st_cnt'] == 0:
            return 'has_sa_not_active_on_1st'
        if r['agr_sa_active_on_1st_cnt'] > 0 and r['trx_dag_cnt_feb'] == 0:
            return 'sa_active_on_1st_but_no_dag_trx'
        if r['agr_sa_active_on_1st_cnt'] > 0 and r['trx_dag_cnt_feb'] > 0:
            return 'sa_active_with_dag_trx'
        return 'other'

    reasons['reason_class'] = reasons.apply(classify, axis=1)
    display(reasons.sort_values(['reason_class', 'inn']))
else:
    print('Нет only_excel INN для классификации')